In [2]:
import cvxpy as cp
import numpy as np
import time

# Read data

In [3]:
def read_data(filepath):
    data = dict()
    data["F"] = []
    data["P"] = []

    with open(filepath,'r') as infile:
        infile.readline()
        data["n"] = int(infile.readline())
        infile.readline()

        infile.readline()
        data["v0"] = int(infile.readline())
        infile.readline()

        infile.readline()
        data["vmin"] = int(infile.readline())
        infile.readline()

        infile.readline()
        data["vmax"] = int(infile.readline())
        infile.readline()

        infile.readline()
        data["tmax"] = int(infile.readline())
        infile.readline()

        infile.readline()
        data["dmax"] = int(infile.readline())
        infile.readline()
        
        infile.readline()
        data["mmax"] = int(infile.readline())
        infile.readline()

        infile.readline()
        data["et"] = float(infile.readline())
        infile.readline()

        infile.readline()
        data["me"] = float(infile.readline())
        infile.readline()

        infile.readline()
        data["tdmin"]= int(infile.readline())
        infile.readline()

        infile.readline()
        data["vtmax"] = int(infile.readline())
        infile.readline()

        infile.readline()
        data["vdmax"] = int(infile.readline())
        infile.readline()

        infile.readline()
        for _ in range(data["n"]):
            data["F"].append(float(infile.readline()))
        infile.readline()

        infile.readline()
        for _ in range(data["n"]):
            data["P"].append(float(infile.readline()))
        
        data["F"] = np.array(data["F"])
        data["P"] = np.array(data["P"])
        
    return data

In [4]:
read_data("BelgiumScenario1_15jours.txt")

{'F': array([ 93975.77028029,  93975.77028029,  93975.77028029,  93975.77028029,
         93975.77028029,  93975.77028029,  93975.77028029,  93975.77028029,
         93975.77028029,  93975.77028029,  93975.77028029,  93975.77028029,
         93975.77028029,  93975.77028029,  93975.77028029,  93975.77028029,
         93975.77028029,  93975.77028029,  93975.77028029,  93975.77028029,
         93975.77028029,  93975.77028029,  93975.77028029,  93975.77028029,
         92945.33447753,  92945.33447753,  92945.33447753,  92945.33447753,
         92945.33447753,  92945.33447753,  92945.33447753,  92945.33447753,
         92945.33447753,  92945.33447753,  92945.33447753,  92945.33447753,
         92945.33447753,  92945.33447753,  92945.33447753,  92945.33447753,
         92945.33447753,  92945.33447753,  92945.33447753,  92945.33447753,
         92945.33447753,  92945.33447753,  92945.33447753,  92945.33447753,
         96654.30561556,  96654.30561556,  96654.30561556,  96654.30561556,
       

# 2.1) and 2.2)

In [5]:
def hydro_with_cost(data, C_jour = 50000):
    params = read_data(data)
    N = int(params['n']) 
    
    
    #Decision variables
    T = cp.Variable(N, nonneg=True, name="Turbinage")
    M = cp.Variable(N, nonneg=True, name="Pompage")
    D = cp.Variable(N, nonneg=True, name="Délestage")
    V = cp.Variable(N, nonneg=True, name="Volume")

    #added binary decision variable
    c = cp.Variable(15, boolean=True, name="Activation de la Turbine (jour)")
    
    #Objective function
    profits = params['P'] @ (params['et'] * T - params['me'] * M)
    #with cost
    if C_jour != 0:
        profits = params['P'] @ (params['et'] * T - params['me'] * M) - (cp.sum(c * C_jour))

    objective = cp.Maximize(profits)
    
    #list of constraints
    constraints = []
    
    constraints += [V[0] == params['v0']]
    #Daily turbine cost constraint
    if C_jour != 0:
        for j in range(15):
            for h in range(24):
                t = j * 24 + h
                constraints.append(T[t] <= params['tmax'] * c[j])
    
    for t in range(N):
        
        #Limits 
        constraints += [V[t] >= params['vmin'], V[t] <= params['vmax']]
        constraints += [T[t] <= params['tmax']]
        constraints += [M[t] <= params['mmax']]
        constraints += [D[t] <= params['dmax']]
        
        #TDlimit
        constraints += [T[t] + D[t] >= params['tdmin']]

        if t > 0:
            #volume reservoir (V0 quand t == 0)
            constraints += [V[t] == V[t-1] + params['F'][t] + M[t] - T[t] - D[t]]
            #Derivative limits
            constraints += [(T[t] - T[t-1]) <= params['vtmax']]
            constraints += [(T[t] - T[t-1]) >= -params['vtmax']]
            constraints += [(D[t] - D[t-1]) <= params['vdmax']]
            constraints += [(D[t] - D[t-1]) >= -params['vdmax']]
            
    #solver
    prob = cp.Problem(objective, constraints)
    prob.solve(solver=cp.HIGHS)
    
    sol = {
        "C": c.value,
        "V": V.value,       
        "T": T.value,       
        "D": D.value,       
        "M": M.value,       
        "valopt": prob.value 
    }
    
    return sol

In [21]:
result = hydro_with_cost("BelgiumScenario1_15jours.txt",150000)
print(f"Bénéfice optimal : {result['valopt']:.2f} €")
result

Bénéfice optimal : 321003.68 €


{'C': array([1., 0., 0., 1., 0., 0., 1., 0., 0., 1., 0., 1., 0., 0., 1.]),
 'V': array([5000000.        , 4893975.77028029, 4937951.54056058,
        4981927.31084087, 5025903.08112116, 5069878.85140145,
        4913854.62168174, 4607830.39196203, 4301806.16224232,
        4145781.93252261, 4189757.7028029 , 4233733.47308319,
        4277709.24336348, 4321685.01364377, 4365660.78392406,
        4909636.55420435, 5442169.60803797, 5486145.37831826,
        5330121.14859855, 5024096.91887884, 4718072.68915913,
        4412048.45943942, 4106024.22971971, 4000000.        ,
        4042945.33447753, 4035890.66895506, 4003640.3223155 ,
        4021389.97567595, 4064335.31015348, 4107280.64463101,
        4150225.97910854, 4193171.31358606, 4236116.64806359,
        4279061.98254112, 4322007.31701865, 4364952.65149618,
        4407897.9859737 , 4450843.32045123, 4493788.65492876,
        4536733.98940629, 4579679.32388382, 4622624.65836134,
        4665569.99283887, 4708515.3273164 , 4751460.

run time

In [7]:
num_runs = 10
run_times = []

for run in range(num_runs):
    start_time = time.time()
    hydro_with_cost("BelgiumScenario1_15jours.txt",0)
    end_time = time.time()
    
    run_times.append(end_time-start_time)

average_time = np.mean(run_times)

print(f"Average execution time: {average_time:.4f} seconds")

Average execution time: 3.1231 seconds


In [22]:
import plotly.graph_objects as ui
from plotly.subplots import make_subplots
import kaleido


def plot_hydro_profits(solutions_dict, data):
    params = read_data(data) if isinstance(data, str) else data
    P = np.array(params["P"])
    et = params["et"]
    me = params["me"]

    fig = make_subplots()

    # Define unique colors for each line to tell them apart easily
    colors = {
        0: "#ff595e",  
        50000: "#b99328",  
        150000: "#A3B9C9",  
    }

    # Track overall maximum hours to set up the axis ranges and day lines
    max_N = 0

    # Iterate through each pre-calculated case passed into the function
    for C_jour, solution in solutions_dict.items():
        T = np.array(solution["T"])
        M = np.array(solution["M"])

        N = len(T)
        max_N = max(max_N, N)
        hours = np.arange(N)

        # Calculate specific profiles for this instance
        hourly_operational_profit = P * (et * T - me * M)
        hourly_profit = hourly_operational_profit.copy()

        # Deduct fixed costs on active days
        for j in range(15):
            day_slice = slice(j * 24, (j + 1) * 24)
            if np.max(T[day_slice]) > 1e-3:
                hourly_profit[day_slice] -= C_jour / 24

        cumulative_profit = np.cumsum(hourly_profit)
        final_profit = cumulative_profit[-1]
        color = colors.get(C_jour, "rgb(127, 127, 127)")

        # Add the line trace for this specific C_jour scenario
        fig.add_trace(
            ui.Scatter(
                x=hours,
                y=cumulative_profit,
                name=f"C_jour = {C_jour:,} €",
                mode="lines",
                line=dict(color=color, width=3),
            ),
        )

        # --- END LABEL FOR EACH LINE ---
        fig.add_annotation(
            x=hours[-1],
            y=final_profit,
            text=f"{final_profit:,.0f} €",
            showarrow=True,
            arrowhead=2,
            arrowcolor=color,
            arrowsize=0.8,
            ax=45,  # Push labels out to the right in the padded area
            ay=0,  # Keep them vertically level with the line tip
            font=dict(size=11, color="white"),
            bgcolor=color,
            bordercolor=color,
            borderwidth=1,
            borderpad=4,
        )

    # --- GENERATE VERTICAL DAY SEGMENT SHAPES ---
    day_lines = []
    for hour in range(24, max_N, 24):
        day_lines.append(
            dict(
                type="line",
                xref="x",
                yref="paper",
                x0=hour,
                y0=0,
                x1=hour,
                y1=1,
                line=dict(
                    color="rgba(100, 100, 100, 0.4)", width=1.5, dash="dash"
                ),
            )
        )

    # --- GLOBAL GRAPH UI TWEAKS ---
    fig.update_layout(
        title="Profit Cumulatif Optimal Selon le Coût Journalier de la Turbine",
        xaxis_title="Temps (heures)",
        legend=dict(
            x=0.01,
            y=0.99,
            bgcolor="rgba(255,255,255,0.85)",
            bordercolor="rgba(0,0,0,0.1)",
            borderwidth=1,
        ),
        template="plotly_white",
        # Extra right side padding (N + 70) to let the labels sit cleanly off the data lines
        xaxis=dict(range=[-10, max_N+14]),
        shapes=day_lines,
    )

    fig.update_xaxes(
        showgrid=True,
        gridcolor="rgba(230, 230, 230, 0.4)",
        dtick=24,
    )

    fig.update_yaxes(title_text="Profit Cumulatif (€)")

    fig.show()
    fig.write_image("2_2_cumulative.pdf", format="pdf", width=800, height=500)

In [23]:
results_dict = {
    0: hydro_with_cost("BelgiumScenario1_15jours.txt", C_jour=0),
    50000: hydro_with_cost("BelgiumScenario1_15jours.txt", C_jour=50000),
    150000: hydro_with_cost("BelgiumScenario1_15jours.txt", C_jour=150000),
}


plot_hydro_profits(results_dict, "BelgiumScenario1_15jours.txt")

In [10]:
result = hydro_with_cost("BelgiumScenario1_15jours.txt",150000)
print(result["C"])

[1. 0. 0. 1. 0. 0. 1. 0. 0. 1. 0. 1. 0. 0. 1.]


In [11]:
def hydro_baseline(data):#flux turbiné compense exactement le flux entrant à chaque instant.
    params = read_data(data)
    return sum([params["P"][t]*params["et"]*params["F"][t] for t in range(params["n"])])

In [12]:
def hydro_rolling_horizon(data_path, H=6):
    params = read_data(data_path)
    N = int(params['n'])
    
    P = params['P']
    F = params['F']
    et = params['et']
    me = params['me']
    
    #To store decision made locally
    T = np.zeros(N)
    M = np.zeros(N)
    D = np.zeros(N)
    V = np.zeros(N)
    
    current_volume = params['v0']
    
    #Local optimisation
    #for every hour
    for t in range(N):
        
        current_H = min(H, N - t) #Deal with end of time
        
        #local decision variables
        T_local = cp.Variable(current_H, nonneg=True, name="Turbinage")
        M_local = cp.Variable(current_H, nonneg=True, name="Pompage")
        D_local = cp.Variable(current_H, nonneg=True, name="Délestage")
        V_local = cp.Variable(current_H, nonneg=True, name="Volume")
        
        #Subset of P and F
        P_local = P[t : t + current_H]
        F_local = F[t : t + current_H]
        
        #Local Objective function
        obj_local = cp.Maximize(P_local @ (et * T_local - me * M_local))
        
        
        constraints = []
        
        #local constraints
        for k in range(current_H):
            constraints += [
                V_local[k] >= params['vmin'], V_local[k] <= params['vmax'],
                T_local[k] <= params['tmax'],
                M_local[k] <= params['mmax'],
                D_local[k] <= params['dmax'],
                T_local[k] + D_local[k] >= params['tdmin']
            ]            
            #Volume update constraint    
            #flow accelration constranit with real decision taken right before
            if k == 0:
                
                if t > 0:
                    constraints += [V_local[0] == current_volume + F_local[0] + M_local[0] - T_local[0] - D_local[0]]
                    constraints += [T_local[0] - T[t-1] <= params['vtmax']]
                    constraints += [T_local[0] - T[t-1] >= -params['vtmax']]
                    constraints += [D_local[0] - D[t-1] <= params['vdmax']]
                    constraints += [D_local[0] - D[t-1] >= -params['vdmax']]
                else:
                    constraints += [V_local[0] == current_volume]


            else:
                constraints += [V_local[k] == V_local[k-1] + F_local[k] + M_local[k] - T_local[k] - D_local[k]]

                constraints += [T_local[k] - T_local[k-1] <= params['vtmax']]
                constraints += [T_local[k] - T_local[k-1] >= -params['vtmax']]
                constraints += [D_local[k] - D_local[k-1] <= params['vdmax']]
                constraints += [D_local[k] - D_local[k-1] >= -params['vdmax']]
                
        #Solving subproblem
        prob = cp.Problem(obj_local, constraints)
        prob.solve(solver=cp.HIGHS)
        
            
        #Apply only first decision
        T[t] = T_local[0].value
        M[t] = M_local[0].value
        D[t] = D_local[0].value
        V[t] = V_local[0].value
        
        # Mise à jour de l'état du réservoir pour le pas de temps suivant 
        current_volume = V[t]

    # Calcul du profit total réel cumulé (avec les décisions prises au jour le jour)
    realized_profits = P * (et * T - me * M)
    total_profit = np.sum(realized_profits)
    
    return {
        "T": T,
        "M": M,
        "D": D,
        "V": V,
        "valopt": total_profit,
        "cumulative_profit": np.cumsum(realized_profits)
    }

In [29]:
results = hydro_rolling_horizon("BelgiumScenario1_15jours.txt",6)
results["valopt"]


np.float64(1715841.114700288)

In [26]:
def plot_horizon_profits(solutions_dict, data):
    params = read_data(data) if isinstance(data, str) else data
    P = np.array(params["P"])
    et = params["et"]
    me = params["me"]

    fig = make_subplots()

    # Define unique colors for each line to tell them apart easily
    colors = {
        "Global": "#ff595e",  
        "Horizon Glissant": "#A3B9C9",  
    }

    # Track overall maximum hours to set up the axis ranges and day lines
    max_N = 0

    # Iterate through each pre-calculated case passed into the function
    for model, solution in solutions_dict.items():
        T = np.array(solution["T"])
        M = np.array(solution["M"])

        N = len(T)
        max_N = max(max_N, N)
        hours = np.arange(N)

        # Calculate specific profiles for this instance
        hourly_operational_profit = P * (et * T - me * M)
        hourly_profit = hourly_operational_profit.copy()

        cumulative_profit = np.cumsum(hourly_profit)
        final_profit = cumulative_profit[-1]
        color = colors.get(model, "rgb(127, 127, 127)")

        # Add the line trace for this specific C_jour scenario
        fig.add_trace(
            ui.Scatter(
                x=hours,
                y=cumulative_profit,
                name=f"Modèle {model}",
                mode="lines",
                line=dict(color=color, width=3),
            ),
        )

        fig.add_annotation(
            x=hours[-1],
            y=final_profit,
            text=f"{final_profit:,.0f} €",
            showarrow=True,
            arrowhead=2,
            arrowcolor=color,
            arrowsize=0.8,
            ax=45,  # Push labels out to the right in the padded area
            ay=0,  # Keep them vertically level with the line tip
            font=dict(size=11, color="white"),
            bgcolor=color,
            bordercolor=color,
            borderwidth=1,
            borderpad=4,
        )

    day_lines = []
    for hour in range(24, max_N, 24):
        day_lines.append(
            dict(
                type="line",
                xref="x",
                yref="paper",
                x0=hour,
                y0=0,
                x1=hour,
                y1=1,
                line=dict(
                    color="rgba(100, 100, 100, 0.4)", width=1.5, dash="dash"
                ),
            )
        )

    # --- GLOBAL GRAPH UI TWEAKS ---
    fig.update_layout(
        title="Profit Cumulatif Optimal des Modèles Global et d'Horizon Glissant (H = 6)",
        xaxis_title="Temps (heures)",
        legend=dict(
            x=0.01,
            y=0.99,
            bgcolor="rgba(255,255,255,0.85)",
            bordercolor="rgba(0,0,0,0.1)",
            borderwidth=1,
        ),
        template="plotly_white",
        # Extra right side padding (N + 70) to let the labels sit cleanly off the data lines
        xaxis=dict(range=[-10, max_N+14]),
        shapes=day_lines,
    )

    fig.update_xaxes(
        showgrid=True,
        gridcolor="rgba(230, 230, 230, 0.4)",
        dtick=24,
    )

    fig.update_yaxes(title_text="Profit Cumulatif (€)")

    fig.show()
    fig.write_image("2_3_cumulative.pdf", format="pdf", width=800, height=500)

In [28]:
results_dict = {
     "Global": hydro_with_cost("BelgiumScenario1_15jours.txt", C_jour=0),
     "Horizon Glissant": hydro_rolling_horizon("BelgiumScenario1_15jours.txt", 6),
 }


plot_horizon_profits(results_dict, "BelgiumScenario1_15jours.txt")

In [ ]:
import plotly.express as px

def plot_all_horizon_profits(solutions_dict, data):
    params = read_data(data) if isinstance(data, str) else data
    P = np.array(params["P"])
    et = params["et"]
    me = params["me"]

    fig = make_subplots()


    # Track overall maximum hours to set up the axis ranges and day lines
    max_N = 0
    num_models = len(solutions_dict)
    legend_items = []

    colors = px.colors.sample_colorscale("turbo", [i / (num_models - 1) for i in range(num_models)])

    # Iterate through each pre-calculated case passed into the function
    for idx, (model, solution) in enumerate(solutions_dict.items()):
        T = np.array(solution["T"])
        M = np.array(solution["M"])

        
        N = len(T)
        max_N = max(max_N, N)
        hours = np.arange(N)

        # Calculate specific profiles for this instance
        hourly_operational_profit = P * (et * T - me * M)
        hourly_profit = hourly_operational_profit.copy()

        cumulative_profit = np.cumsum(hourly_profit)
        final_profit = cumulative_profit[-1]

        # Add the line trace for this specific C_jour scenario
        fig.add_trace(
            ui.Scatter(
                x=hours,
                y=cumulative_profit,
                name=f"H= {model}",
                mode="lines",
                line=dict(
                    width=2,  # Slightly thinner helps reduce spaghetti overlap
                    color=colors[idx],  # Assign distinct color
                ),
            ),
        )
        colored_marker = f'<span style="color:{colors[idx]}; font-size:12px;">●</span> H={model}'
        legend_items.append(colored_marker)

        if model<9:
            fig.add_annotation(
                x=N,
                y=final_profit,
                text=f"H={model}",
                showarrow=False,
                xanchor="left",
                yanchor="middle",
                font=dict(color=colors[idx], size=10),
            )

    day_lines = []
    for hour in range(24, max_N, 24):
        day_lines.append(
            dict(
                type="line",
                xref="x",
                yref="paper",
                x0=hour,
                y0=0,
                x1=hour,
                y1=1,
                line=dict(
                    color="rgba(100, 100, 100, 0.4)", width=1.5, dash="dash"
                ),
            )
        )

    # --- GLOBAL GRAPH UI TWEAKS ---
    fig.update_layout(
        title="Profit Cumulatif des Modèles d'Horizon Glissant par taille de l'horizon",
        xaxis_title="Temps (heures)",
        template="plotly_white",
        # Extra right side padding (N + 70) to let the labels sit cleanly off the data lines
        xaxis=dict(range=[144, max_N]),
        shapes=day_lines,
        showlegend=False
    )

    fig.update_xaxes(
        showgrid=True,
        gridcolor="rgba(230, 230, 230, 0.4)",
        dtick=24,
    )

    halfway = len(legend_items) // 2
    col1_items = legend_items[:halfway]
    col2_items = legend_items[halfway:]

    # Join them with vertical line breaks (<br>) to make two vertical lists
    col1_html = "<br>".join(col1_items)
    col2_html = "<br>".join(col2_items)

    # --- ANNOTATION FOR COLUMN 1 (Left Side) ---
    fig.add_annotation(
        xref="paper",
        yref="paper",
        x=0.02,          # Positions it on the left third of the plot
        y=1,         # Pushes it cleanly below the X-axis label
        text=col1_html,
        showarrow=False,
        xanchor="left",  # Forces the bounding box to anchor from its left edge
        yanchor="top",
        align="left",    # Left-aligns the text inside
        font=dict(size=9, color="#333"),
    )

    # --- ANNOTATION FOR COLUMN 2 (Right Side) ---
    fig.add_annotation(
        xref="paper",
        yref="paper",
        x=0.12,          # Positions it on the right third of the plot
        y=1,         # Perfectly aligned vertically with Column 1
        text=col2_html,
        showarrow=False,
        xanchor="left",  # Forces the bounding box to anchor from its left edge
        yanchor="top",
        align="left",    # Left-aligns the text inside
        font=dict(size=9, color="#333"),
    )

    fig.update_yaxes(title_text="Profit Cumulatif (€)")

    fig.show()
    fig.write_image("2_4_cumulative.pdf", format="pdf", width=800, height=500)

In [ ]:
horizons = [2,3,4,5,6,8,12,18,24,36,48,64,128,360]
rolling_horizons_results = dict()
for h in horizons:
    rolling_horizons_results[h] = hydro_rolling_horizon("BelgiumScenario1_15jours.txt",h)
    print("Done",h)

In [ ]:
plot_all_horizon_profits(rolling_horizons_results,"BelgiumScenario1_15jours.txt")